# zander-ml — Colab training

One matrix run per session step: clone the repo, pull the tiles zip from Drive, train, and drop the run **bundle** back on Drive. Import it locally with `python cli.py import-run <bundle.zip>` and evaluate with `python cli.py eval`.

Prereqs on your Drive (`ZanderML/`): `ds_v1_tiles.zip` (build locally: `python cli.py tile --zip`, ~1 GB).

Runtime: **T4 GPU** (Runtime → Change runtime type).

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ZanderML'

In [ ]:
# Clone the repo (public GitHub — no token needed) and install deps.
!git clone --branch cv-project-on-pc https://github.com/VictoRoferz/zander.git /content/zander
%pip -q install -r /content/zander/ml/requirements.txt

In [ ]:
# Dataset: unzip the tiles from Drive, then rebuild the symlinked YOLO view.
import os, zipfile, pathlib
ML_DATA = '/content/ml_data'
os.environ['ML_DATA_ROOT'] = ML_DATA
pathlib.Path(f'{ML_DATA}/tiles').mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(f'{DRIVE}/ds_v1_tiles.zip') as zf:
    zf.extractall(f'{ML_DATA}/tiles')
%cd /content/zander/ml
!python cli.py tile --no-images   # COCO GT + YOLO view from the unzipped tiles
!ls {ML_DATA}/yolo/ds_v1 | head

In [ ]:
# Train one config (edit CONFIG per run; ~10 GPU-min per fold-run on a T4).
CONFIG = 'configs/matrix/yolo11s_fold0.yaml'
%cd /content/zander/ml
!python cli.py train --config {CONFIG} --device 0

In [ ]:
# Option B (Colab Pro): run the WHOLE matrix in one session (~1.5-2 h on T4/L4).
# Each bundle is copied to Drive right after its run finishes, so a mid-loop
# disconnect loses at most the run in progress. Skip the single-run cell above.
import glob, os, shutil, subprocess
%cd /content/zander/ml
for cfg in sorted(glob.glob('configs/matrix/*.yaml')):
    print(f'=== {cfg} ===', flush=True)
    subprocess.run(['python', 'cli.py', 'train', '--config', cfg, '--device', '0'], check=True)
    for bundle in glob.glob(f'{os.environ["ML_DATA_ROOT"]}/run_dirs/*_bundle.zip'):
        print('->', shutil.copy(bundle, DRIVE), flush=True)
print('matrix complete')

In [ ]:
# Push the bundle back to Drive.
import glob, shutil, os
for bundle in glob.glob(f'{os.environ["ML_DATA_ROOT"]}/run_dirs/*_bundle.zip'):
    print('->', shutil.copy(bundle, DRIVE))